## Phase 2.5 – Manual Cluster Verification and Sensor Selection


---
# Step : Load Clustered FD001 Data 

In [1]:
import pandas as pd

# Load clustered FD001 dataset
df_fd001 = pd.read_csv("../data/clustered_train_FD001.csv")

# Step 1: Identify all sensor columns
sensor_columns_fd001 = [col for col in df_fd001.columns if col.startswith('sensor_')]

# Step 2: Drop sensors with extremely low standard deviation (essentially flat)
flat_sensors_fd001 = [s for s in sensor_columns_fd001 if df_fd001[s].std() < 0.001]
usable_sensors_fd001 = [s for s in sensor_columns_fd001 if s not in flat_sensors_fd001]

# Step 3: Compute variances (for information only)
variances_fd001 = df_fd001[usable_sensors_fd001].var().sort_values(ascending=False)

# Final Output
print("Flat / Constant Sensors dropped:", flat_sensors_fd001)
print(f"Usable Sensors retained for FD001 (total {len(usable_sensors_fd001)}):")
print(usable_sensors_fd001)


Flat / Constant Sensors dropped: []
Usable Sensors retained for FD001 (total 15):
['sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


# Step: Plot Sensor Behavior Across Cluster Stages (FD001)

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load clustered FD001 dataset
df_fd001 = pd.read_csv("../data/clustered_train_FD001.csv")

# Identify usable sensors
sensor_columns = [col for col in df_fd001.columns if col.startswith('sensor_')]
usable_sensors = [s for s in sensor_columns if df_fd001[s].std() >= 0.001]

# Output directory
output_dir = "../figures/manual_cluster_verification/FD001/"
os.makedirs(output_dir, exist_ok=True)

# Seaborn settings
sns.set(style="whitegrid")

# Plot each sensor across kmeans_cluster stages
for sensor in usable_sensors:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='kmeans_cluster',  
        data=df_fd001,
        palette='tab10'
    )
    plt.title(f"{sensor} Behavior Across KMeans Clusters - FD001")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel(f"{sensor} Reading (Normalized)")
    plt.legend(title="KMeans Cluster")
    plt.tight_layout()

    # Save plot
    filename = f"{sensor}_kmeans_cluster_FD001.png"
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

print(" Sensor behavior plots saved using 'kmeans_cluster' as hue.")


 Sensor behavior plots saved using 'kmeans_cluster' as hue.


### Stage-Time Summary Check

In [3]:
import pandas as pd

# Load your corrected clustered data
df = pd.read_csv("../data/corrected_clustered_train_FD001.csv")

# Group by final_stage and compute average cycle
avg_time_per_stage = df.groupby('final_stage')['time'].mean().sort_index()

print("📊 Average cycle (time) per final_stage:")
print(avg_time_per_stage)


📊 Average cycle (time) per final_stage:
final_stage
0    203.344167
1    191.846308
2    132.547953
3     86.077242
4     62.355054
Name: time, dtype: float64


### Final stage behavior plots

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

usable_sensors = [col for col in df.columns if col.startswith('sensor_') and df[col].std() >= 0.001]
output_dir = "../figures/manual_cluster_verification/final_stage/final_stage_FD001"
os.makedirs(output_dir, exist_ok=True)

for sensor in usable_sensors:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='final_stage',
        data=df,
        palette='tab10',
        errorbar=None
    )
    plt.title(f"{sensor} Behavior Across Final Stages - FD001")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel(f"{sensor} Reading (Normalized)")
    plt.legend(title="Final Stage")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{sensor}_final_stage.png"))
    plt.close()

print("✅ Final stage behavior plots saved.")


✅ Final stage behavior plots saved.


### FD001: Cluster-to-Stage Mapping Verification

In [4]:
import pandas as pd

df_fd001 = pd.read_csv("../data/clustered_train_FD001.csv")

# Step 1: Average cycle time per cluster
avg_times_fd001 = df_fd001.groupby("kmeans_cluster")["time"].mean().sort_values(ascending=False)

# Step 2: Assign stage numbers in descending order
sorted_clusters_fd001 = avg_times_fd001.index.tolist()
correct_mapping_fd001 = {cluster: stage for stage, cluster in enumerate(sorted_clusters_fd001)}

print("✅ FD001 Verified Mapping:\n", correct_mapping_fd001)


✅ FD001 Verified Mapping:
 {2: 0, 3: 1, 0: 2, 4: 3, 1: 4}


In [6]:
import pandas as pd

# Load clustered FD001 dataset
df_fd001 = pd.read_csv("../data/clustered_train_FD001.csv")

# ✅ Define your manual cluster-to-stage mapping here
cluster_to_stage_fd001 = {
    2: 0,
    3: 1,
    0: 2,
    4: 3,
    1: 4
}

# 🪄 Apply mapping to create final_stage column
df_fd001['final_stage'] = df_fd001['kmeans_cluster'].map(cluster_to_stage_fd001)

# 💾 Save the corrected clustered file for Phase 3 usage
df_fd001.to_csv("../data/corrected_clustered_train_FD001.csv", index=False)

# ✅ Optional: Save stage mapping reference
mapping_df = pd.DataFrame(list(cluster_to_stage_fd001.items()), columns=['kmeans_cluster', 'final_stage'])
mapping_df.to_csv("../report/fd001_cluster_stage_mapping.csv", index=False)

print("✔ Final stage labels assigned and saved for FD001.")

✔ Final stage labels assigned and saved for FD001.


 ---
 # Step : Load FD002 Clustered Data 

In [7]:
import pandas as pd

# Load clustered FD002 dataset
df_fd002 = pd.read_csv("../data/clustered_train_FD002.csv")

# Step 1: Identify all sensor columns
sensor_columns_fd002 = [col for col in df_fd002.columns if 'sensor_' in col]

# Step 2: Drop sensors with extremely low std dev (flat/noise)
flat_sensors_fd002 = [s for s in sensor_columns_fd002 if df_fd002[s].std() < 0.001]
usable_sensors_fd002 = [s for s in sensor_columns_fd002 if s not in flat_sensors_fd002]

# Output summaries
print("Flat / Constant Sensors dropped:", flat_sensors_fd002)
print(f"Usable Sensors retained for FD002 (total {len(usable_sensors_fd002)}):")
print(usable_sensors_fd002)

Flat / Constant Sensors dropped: []
Usable Sensors retained for FD002 (total 21):
['sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


# Step – Plot Behavior by Cluster Stage (FD002)

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load clustered FD002 dataset
df_fd002 = pd.read_csv("../data/clustered_train_FD002.csv")

# Identify usable sensor columns
sensor_columns = [col for col in df_fd002.columns if 'sensor_' in col]
usable_sensors = [s for s in sensor_columns if df_fd002[s].std() >= 0.001]

# Output directory
output_dir = "../figures/manual_cluster_verification/FD002/"
os.makedirs(output_dir, exist_ok=True)

# Plot settings
sns.set(style="whitegrid")

# ✅ FIX: use 'kmeans_cluster' instead of missing 'kmeans_stage'
for sensor in usable_sensors:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='kmeans_cluster',
        data=df_fd002,
        palette='tab10',
        legend='full'
    )
    plt.title(f"{sensor} Behavior Across KMeans Clusters - FD002")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel(f"{sensor} Reading (Normalized)")
    plt.legend(title='KMeans Cluster')
    plt.tight_layout()
    plt.savefig(f"{output_dir}{sensor}_kmeans_cluster_FD002.png")
    plt.close()

print("✅ All sensor behavior plots saved for FD002 using kmeans_cluster.")


✅ All sensor behavior plots saved for FD002 using kmeans_cluster.


### Stage-Time Summary Check (FD002)

In [9]:
import pandas as pd

# Load FD002 with final_stage
df_fd002 = pd.read_csv("../data/corrected_clustered_train_FD002.csv")

# Compute average cycle time per final_stage
avg_time_per_stage_fd002 = df_fd002.groupby("final_stage")["time"].mean().round(2)

# Record count per stage (optional check)
count_per_stage_fd002 = df_fd002["final_stage"].value_counts().sort_index()

print("📊 Average Cycle Time per Final Stage (FD002):")
print(avg_time_per_stage_fd002)

print("\n📦 Record Count per Final Stage (FD002):")
print(count_per_stage_fd002)


📊 Average Cycle Time per Final Stage (FD002):
final_stage
0    121.41
1    109.99
2    109.03
3    108.95
4     89.90
Name: time, dtype: float64

📦 Record Count per Final Stage (FD002):
final_stage
0     9836
1     8002
2    21495
3     8044
4     6382
Name: count, dtype: int64


### Final Stage Behavior Plots (FD002)

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Reuse dataframe
usable_sensors_fd002 = [col for col in df_fd002.columns if col.startswith('sensor_') and df_fd002[col].std() >= 0.001]

# Output directory for final_stage plots
output_dir = "../figures/manual_cluster_verification/final_stage/final_stage_FD002/"
os.makedirs(output_dir, exist_ok=True)

# Seaborn style
sns.set(style="whitegrid")

# Generate and save plots
for sensor in usable_sensors_fd002:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='final_stage',
        data=df_fd002,
        palette='tab10',
        errorbar=None  # use errorbar=None instead of deprecated ci=None
    )
    plt.title(f"{sensor} Behavior Across Final Stages - FD002")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel(f"{sensor} Reading (Normalized)")
    plt.legend(title="Final Stage")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{sensor}_final_stage_FD002.png"))
    plt.close()

print("✅ Final stage behavior plots saved for FD002.")


✅ Final stage behavior plots saved for FD002.


### FD002: Cluster-to-Stage Mapping Verification

In [11]:
import pandas as pd

df_fd002 = pd.read_csv("../data/clustered_train_FD002.csv")

# Step 1: Average cycle time per cluster
avg_times_fd002 = df_fd002.groupby("kmeans_cluster")["time"].mean().sort_values(ascending=False)

# Step 2: Assign stage numbers in descending order
sorted_clusters_fd002 = avg_times_fd002.index.tolist()
correct_mapping_fd002 = {cluster: stage for stage, cluster in enumerate(sorted_clusters_fd002)}

print("✅ FD002 Verified Mapping:\n", correct_mapping_fd002)


✅ FD002 Verified Mapping:
 {3: 0, 2: 1, 0: 2, 1: 3, 4: 4}


# Step: Add final_stage and Save

In [12]:
import pandas as pd

# Load clustered FD002 dataset
df_fd002 = pd.read_csv("../data/clustered_train_FD002.csv")

# ✅ Define your manual cluster-to-stage mapping here
cluster_to_stage_fd002 = {
    3: 0,  # Healthy
    2: 1,
    0: 2,
    1: 3,
    4: 4   # Failure
}

# 🪄 Apply mapping to create final_stage column
df_fd002['final_stage'] = df_fd002['kmeans_cluster'].map(cluster_to_stage_fd002)

# 💾 Save the corrected clustered file for Phase 3 usage
df_fd002.to_csv("../data/corrected_clustered_train_FD002.csv", index=False)

# ✅ Optional: Save stage mapping reference
mapping_df = pd.DataFrame(list(cluster_to_stage_fd002.items()), columns=['kmeans_cluster', 'final_stage'])
mapping_df.to_csv("../report/fd002_cluster_stage_mapping.csv", index=False)

print("✔ Final stage labels assigned and saved for FD002.")


✔ Final stage labels assigned and saved for FD002.


---
# Step : Load FD003 Clustered Data 

In [13]:
import pandas as pd

# Load clustered FD003 dataset
df_fd003 = pd.read_csv("../data/clustered_train_FD003.csv")

# Identify all sensor columns
sensor_columns_fd003 = [col for col in df_fd003.columns if 'sensor_' in col]

# Drop sensors with near-zero standard deviation (flat sensors)
flat_sensors_fd003 = [s for s in sensor_columns_fd003 if df_fd003[s].std() < 0.001]
usable_sensors_fd003 = [s for s in sensor_columns_fd003 if s not in flat_sensors_fd003]

# Compute variances (for reference/logging, not filtering)
variances_fd003 = df_fd003[usable_sensors_fd003].var().sort_values(ascending=False)

#  Output
print("Flat / Constant Sensors dropped:", flat_sensors_fd003)
print(f"Usable Sensors retained for FD003 (total {len(usable_sensors_fd003)}):")
print(usable_sensors_fd003)


Flat / Constant Sensors dropped: []
Usable Sensors retained for FD003 (total 16):
['sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


#  Step – Plot Sensor Behavior Across KMeans Cluster Stages (FD003)

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load clustered FD003 dataset
df_fd003 = pd.read_csv("../data/clustered_train_FD003.csv")

# Ensure required column exists
if 'kmeans_cluster' not in df_fd003.columns:
    raise ValueError("'kmeans_cluster' column not found in FD003 dataset.")

# Identify usable sensor columns (drop near-constant ones)
sensor_cols = [col for col in df_fd003.columns if col.startswith("sensor_")]
usable_sensors = [col for col in sensor_cols if df_fd003[col].std() > 0.001]

# Create output directory
output_dir = "../figures/manual_cluster_verification/FD003/"
os.makedirs(output_dir, exist_ok=True)

# Plot settings
sns.set(style="whitegrid")

# Plot each sensor's behavior
for sensor in usable_sensors:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='kmeans_cluster',
        data=df_fd003,
        palette='tab10',
        legend='full'
    )
    plt.title(f"{sensor} Behavior Across KMeans Stages - FD003")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel("Normalized Sensor Reading")
    plt.legend(title='KMeans Stage')
    plt.tight_layout()
    plt.savefig(f"{output_dir}{sensor}_kmeans_stage_FD003.png")
    plt.close()

print(" Sensor behavior plots saved for FD003 (all usable sensors).")


 Sensor behavior plots saved for FD003 (all usable sensors).


### Stage-Time Summary Check (FD003)

In [15]:
import pandas as pd

# Load corrected FD003 with final_stage
df_fd003 = pd.read_csv("../data/corrected_clustered_train_FD003.csv")

# Compute average cycle time and count per stage
avg_time_per_stage = df_fd003.groupby("final_stage")["time"].mean().round(2)
stage_counts = df_fd003["final_stage"].value_counts().sort_index()

print("📊 Average Cycle Time per Final Stage (FD003):")
print(avg_time_per_stage)

print("\n📦 Record Count per Final Stage (FD003):")
print(stage_counts)


📊 Average Cycle Time per Final Stage (FD003):
final_stage
0    310.89
1    269.68
2    163.59
3    130.99
4     91.17
Name: time, dtype: float64

📦 Record Count per Final Stage (FD003):
final_stage
0     1191
1     1741
2     4571
3     7052
4    10165
Name: count, dtype: int64


### Final Stage Behavior Plots (FD003)

In [16]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Use same DataFrame
usable_sensors = [col for col in df_fd003.columns if col.startswith("sensor_") and df_fd003[col].std() > 0.001]

# Output folder
output_dir = "../figures/manual_cluster_verification/final_stage/final_stage_FD003/"
os.makedirs(output_dir, exist_ok=True)

# Plot settings
sns.set(style="whitegrid")

# Plot sensor trends colored by final_stage
for sensor in usable_sensors:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='final_stage',
        data=df_fd003,
        palette='tab10',
        errorbar=None  # replace ci=None
    )
    plt.title(f"{sensor} Behavior Across Final Stages - FD003")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel("Normalized Sensor Reading")
    plt.legend(title="Final Stage")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"{sensor}_final_stage_FD003.png"))
    plt.close()

print("✅ Final stage behavior plots saved for FD003.")


✅ Final stage behavior plots saved for FD003.


### FD002: Cluster-to-Stage Mapping Verification

In [17]:
import pandas as pd

df = pd.read_csv("../data/clustered_train_FD003.csv")

# Step 1: Average cycle time per cluster
avg_times = df.groupby("kmeans_cluster")["time"].mean().sort_values(ascending=False)

# Step 2: Assign stage numbers in descending order
sorted_clusters = avg_times.index.tolist()
correct_mapping = {cluster: stage for stage, cluster in enumerate(sorted_clusters)}

print("✅ Verified Correct Mapping:\n", correct_mapping)


✅ Verified Correct Mapping:
 {0: 0, 3: 1, 2: 2, 1: 3, 4: 4}


# Step: Add final_stage and Save

In [18]:
import pandas as pd

# Load clustered FD003 dataset
df_fd003 = pd.read_csv("../data/clustered_train_FD003.csv")

# ✅ Correct cluster-to-stage mapping
correct_mapping_fd003 = {
    0: 0,
    3: 1,
    2: 2,
    1: 3,
    4: 4
}

# Apply mapping
df_fd003["final_stage"] = df_fd003["kmeans_cluster"].map(correct_mapping_fd003)

# Save corrected file
df_fd003.to_csv("../data/corrected_clustered_train_FD003.csv", index=False)

# Save mapping
pd.DataFrame(list(correct_mapping_fd003.items()), columns=["kmeans_cluster", "final_stage"])\
  .to_csv("../report/fd003_cluster_stage_mapping.csv", index=False)

print("✅ Correct final_stage mapping applied and saved for FD003.")


✅ Correct final_stage mapping applied and saved for FD003.


 ---
 # Step : Load FD004 Clustered Data 

In [19]:
import pandas as pd

# Load clustered FD004 dataset
df_fd004 = pd.read_csv("../data/clustered_train_FD004.csv")

# Identify all sensor columns
sensor_columns_fd004 = [col for col in df_fd004.columns if 'sensor_' in col]

# Drop flat sensors (std deviation very close to 0)
flat_sensors_fd004 = [s for s in sensor_columns_fd004 if df_fd004[s].std() < 0.001]
usable_sensors_fd004 = [s for s in sensor_columns_fd004 if s not in flat_sensors_fd004]

# Compute variance (for reference)
variances_fd004 = df_fd004[usable_sensors_fd004].var().sort_values(ascending=False)

# Output results
print("Flat / Constant Sensors dropped:", flat_sensors_fd004)
print(f"Usable Sensors retained for FD004 (total {len(usable_sensors_fd004)}):")
print(usable_sensors_fd004)


Flat / Constant Sensors dropped: []
Usable Sensors retained for FD004 (total 21):
['sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


# Step: Plot Sensor Behavior Across KMeans Stages (FD004)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load clustered FD004 dataset
df_fd004 = pd.read_csv("../data/clustered_train_FD004.csv")

# Safety check
if 'kmeans_cluster' not in df_fd004.columns:
    raise ValueError("'kmeans_cluster' column not found in FD004 dataset.")

# Identify all usable sensor columns (drop flat ones)
sensor_columns_fd004 = [col for col in df_fd004.columns if col.startswith('sensor_')]
usable_sensors_fd004 = [s for s in sensor_columns_fd004 if df_fd004[s].std() >= 0.001]

# Output path
output_dir = "../figures/manual_cluster_verification/FD004/"
os.makedirs(output_dir, exist_ok=True)

# Plot styling
sns.set(style="whitegrid")

# Plot sensor behavior for each usable sensor
for sensor in usable_sensors_fd004:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='kmeans_cluster',
        data=df_fd004,
        palette='tab10'
    )
    plt.title(f"{sensor} behavior across KMeans Stages - FD004")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel(f"{sensor} Reading (Normalized)")
    plt.legend(title='KMeans Stage')
    plt.tight_layout()
    plt.savefig(f"{output_dir}{sensor}_kmeans_stage_FD004.png")
    plt.close()

print("✅ All sensor behavior plots saved for FD004.")


## Cluster-to-Stage Mapping Verification

In [21]:
import pandas as pd

df_fd004 = pd.read_csv("../data/clustered_train_FD004.csv")

# Step 1: Average cycle time per cluster
avg_times_fd004 = df_fd004.groupby("kmeans_cluster")["time"].mean().sort_values(ascending=False)

# Step 2: Create verified mapping (sorted in descending health)
sorted_clusters_fd004 = avg_times_fd004.index.tolist()
correct_mapping_fd004 = {cluster: stage for stage, cluster in enumerate(sorted_clusters_fd004)}

print("✅ FD004 Verified Cluster-to-Stage Mapping:\n", correct_mapping_fd004)


✅ FD004 Verified Cluster-to-Stage Mapping:
 {2: 0, 3: 1, 0: 2, 1: 3, 4: 4}


# Step: Add final_stage and Save

In [22]:
import pandas as pd

# Load the clustered FD004 dataset
df_fd004 = pd.read_csv("../data/clustered_train_FD004.csv")

# Manual relabeling map: kmeans_cluster → Stage #
manual_mapping_fd004 = {
    2: 0,  # Healthy
    3: 1,
    0: 2,
    1: 3,
    4: 4   # Failure
}

# Apply the mapping
df_fd004["final_stage"] = df_fd004["kmeans_cluster"].map(manual_mapping_fd004)

# Save the corrected file
df_fd004.to_csv("../data/corrected_clustered_train_FD004.csv", index=False)
print("[✔] Saved: corrected_clustered_train_FD004.csv with manually relabeled 'final_stage'")


[✔] Saved: corrected_clustered_train_FD004.csv with manually relabeled 'final_stage'


## Stage-Time Summary Check (after you apply final_stage)

In [23]:
import pandas as pd

# Load corrected dataset with final_stage mapping
df_fd004 = pd.read_csv("../data/corrected_clustered_train_FD004.csv")

# Average time per stage
avg_time_per_stage_fd004 = df_fd004.groupby("final_stage")["time"].mean().round(2)
stage_counts_fd004 = df_fd004["final_stage"].value_counts().sort_index()

print("📊 Average Cycle Time per Final Stage (FD004):")
print(avg_time_per_stage_fd004)

print("\n📦 Record Count per Final Stage (FD004):")
print(stage_counts_fd004)


📊 Average Cycle Time per Final Stage (FD004):
final_stage
0    145.23
1    134.70
2    134.27
3    134.06
4    118.99
Name: time, dtype: float64

📦 Record Count per Final Stage (FD004):
final_stage
0    10683
1     9139
2    24557
3     9238
4     7632
Name: count, dtype: int64


##  Final Stage Sensor Behavior Plots (after mapping)

In [24]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pandas as pd

# Load corrected dataset
df_fd004 = pd.read_csv("../data/corrected_clustered_train_FD004.csv")

# Identify usable sensors
usable_sensors_fd004 = [col for col in df_fd004.columns if col.startswith("sensor_") and df_fd004[col].std() > 0.001]

# Output path
output_dir = "../figures/manual_cluster_verification/final_stage/final_stage_FD004/"
os.makedirs(output_dir, exist_ok=True)

# Plotting
sns.set(style="whitegrid")

for sensor in usable_sensors_fd004:
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        x='time',
        y=sensor,
        hue='final_stage',
        data=df_fd004,
        palette='tab10',
        errorbar=None
    )
    plt.title(f"{sensor} behavior across Final Stages - FD004")
    plt.xlabel("Cycle Number (Time)")
    plt.ylabel("Normalized Sensor Reading")
    plt.legend(title="Final Stage")
    plt.tight_layout()
    plt.savefig(f"{output_dir}{sensor}_final_stage_FD004.png")
    plt.close()

print("✅ Final stage behavior plots saved for FD004.")


✅ Final stage behavior plots saved for FD004.


---

In [25]:
import pandas as pd
from sklearn.cluster import KMeans
import os

# Base paths
base_path = "../data"
output_path = "../data/test_stage_predictions"
os.makedirs(output_path, exist_ok=True)

# Dataset list
dataset_ids = ["FD001", "FD002", "FD003", "FD004"]

# ✅ Final verified cluster-to-stage mappings
cluster_to_stage_map = {
    "FD001": {2: 0, 3: 1, 0: 2, 4: 3, 1: 4},
    "FD002": {3: 0, 2: 1, 0: 2, 1: 3, 4: 4},
    "FD003": {0: 0, 3: 1, 2: 2, 1: 3, 4: 4},
    "FD004": {2: 0, 3: 1, 0: 2, 1: 3, 4: 4}
}

# Loop through all FD00x datasets
for ds_id in dataset_ids:
    print(f"\n📦 Processing {ds_id}...")

    # Load cleaned train and test datasets
    train_df = pd.read_csv(os.path.join(base_path, f"clean_train_{ds_id}.csv"))
    test_df  = pd.read_csv(os.path.join(base_path, f"clean_test_{ds_id}.csv"))

    # Get sensor columns (start with 'sensor_')
    sensor_cols = [col for col in train_df.columns if col.startswith("sensor_")]

    # Fit KMeans on train data only
    kmeans = KMeans(n_clusters=5, random_state=42, n_init="auto")
    kmeans.fit(train_df[sensor_cols])

    # Predict cluster labels for test data
    test_df["kmeans_cluster"] = kmeans.predict(test_df[sensor_cols])

    # Map clusters to final degradation stage
    test_df["final_stage"] = test_df["kmeans_cluster"].map(cluster_to_stage_map[ds_id])

    # Save output
    output_file = os.path.join(output_path, f"final_stage_test_{ds_id}.csv")
    test_df.to_csv(output_file, index=False)
    print(f"✅ Saved: {output_file}")



📦 Processing FD001...
✅ Saved: ../data/test_stage_predictions\final_stage_test_FD001.csv

📦 Processing FD002...
✅ Saved: ../data/test_stage_predictions\final_stage_test_FD002.csv

📦 Processing FD003...
✅ Saved: ../data/test_stage_predictions\final_stage_test_FD003.csv

📦 Processing FD004...
✅ Saved: ../data/test_stage_predictions\final_stage_test_FD004.csv
